# Lab 1 — Loan Application Evaluation
### Claude Agent SDK + Messages API

In this lab you will score a commercial loan application narrative against the
**Five C's of Credit**, twice, using two different Claude surfaces:

| | Surface | What it does |
|---|---|---|
| **Part A** | Messages API | One call. You specify the task completely; Claude returns structured JSON. |
| **Part B** | Claude Agent SDK | An agent with a folder and tools. It decides its own steps and writes the memo itself. |

Both parts score the same application. The point of the lab is the **difference in shape** —
when a single call is enough, and when you actually need an agent.

---

**Your Anthropic API key is requested at run time.** It is never written to this
notebook, never committed to the repository, and never stored on disk.

## Step 1 — Clone the repository and install dependencies

Run the cell below. If you see a **Restart session** dialog while it installs,
click **Restart session**, then run this cell again.

In [ ]:
# Clone only what this lab needs
!git clone --depth 1 --filter=blob:none --sparse https://github.com/chetankumarmk56/Claude-Agentic-SDK-Labs.git
%cd Claude-Agentic-SDK-Labs
!git sparse-checkout set Lab-1
%cd /content/Claude-Agentic-SDK-Labs/Lab-1

# Install the lab dependencies
!pip install -q -r requirements.txt

# ngrok exposes the Streamlit app so you can open it from Colab
!pip install -q pyngrok

print("Dependencies installed.")

## Step 2 — Enter your Anthropic API key

`getpass` hides what you type, so the key never appears in the notebook output
and is never saved with the file.

Get a key from [console.anthropic.com](https://console.anthropic.com/settings/keys).

In [ ]:
import os, getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
print("Key stored for this session only.")

## Step 3 — Check the key works

One cheap call, before the lab spends real tokens.

In [ ]:
import sys
sys.path.insert(0, ".")

from src.validate import validate_anthropic_key

print(validate_anthropic_key(os.environ["ANTHROPIC_API_KEY"]))

## Step 4 — Part A: score the application with one Messages API call

The task here is fully specified — read this narrative, apply this rubric,
return these fields. That is exactly what a single call is good at.

In [ ]:
import json
from pathlib import Path

from src.model import score_application
from src.scoring import load_rubric, aggregate

rubric = load_rubric()
narrative = Path("data/strong-approve.txt").read_text(encoding="utf-8")

result = score_application(narrative, rubric, os.environ["ANTHROPIC_API_KEY"])
print(json.dumps(result, indent=2)[:1500])

### The arithmetic

Claude chose the 1–5 score for each C. The weighting, the total and the decision
band are computed in plain Python, so the maths is identical every run.

In [ ]:
summary = aggregate(result["scores"], rubric)

for row in summary["rows"]:
    score = "N/E" if row["score"] is None else row["score"]
    points = "—" if row["points"] is None else round(row["points"], 1)
    print(f"{row['id']}  {row['name']:<12} score={score:<4} weight={row['weight']:<4} points={points}")

print()
print(f"{summary['earned']} / {summary['evidenced_weight']} x 100 = {summary['raw_total']} -> {summary['total']}")
print(f"DECISION: {summary['decision']}")
if summary["overridden_by"]:
    print(f"(set by hard rule {summary['overridden_by']})")

### Try the other samples

Four narratives ship with the lab, and between them they hit every decision band:

| File | What it tests |
|---|---|
| `strong-approve.txt` | Every C evidenced, comfortable metrics |
| `hard-rule-decline.txt` | DSCR below 1.00x — the hard rule should override the band |
| `incomplete-evidence.txt` | No collateral evidence at all — one C should score N/E |
| `stated-vs-computed-conflict.txt` | Claims a DSCR its own figures contradict |

Change the filename above and re-run. Watch what the hard rule does.

## Step 5 — Part B: the same job, given to an agent

Now hand the application to the **Claude Agent SDK**. Instead of one specified
call, the agent gets a folder and its built-in tools, and decides its own steps:
read the rubric, read the narrative, work through the five C's, write a memo.

Watch the `→ Using tool:` lines — that is the agent loop, made visible.

In [ ]:
import asyncio, shutil
from pathlib import Path

from src.agent import run_agent

# Give the agent a clean folder holding only what it needs
workspace = Path("outputs")
workspace.mkdir(exist_ok=True)
shutil.copy("rubric.json", workspace / "rubric.json")
shutil.copy("data/strong-approve.txt", workspace / "strong-approve.txt")

async def main():
    async for kind, text in run_agent(workspace / "strong-approve.txt", workspace):
        prefix = {"tool": "-> ", "result": "[done] ", "text": ""}[kind]
        print(prefix + text[:400])

await main()

### Read what the agent wrote

In [ ]:
from IPython.display import Markdown, display

memo = Path("outputs/credit_memo.md")
display(Markdown(memo.read_text(encoding="utf-8")) if memo.exists()
        else Markdown("_The agent did not write a memo. Re-run the cell above._"))

## Step 6 — Run the web application

Both parts, behind a Streamlit interface. The app prompts for the API key in a
password field — the same run-time pattern, in a UI.

Run the cell, then open the printed ngrok URL.

In [ ]:
from pyngrok import ngrok
import getpass, time, subprocess

# ngrok needs its own free authtoken — https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(getpass.getpass("Enter your ngrok authtoken: "))

ngrok.kill()
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(6)

print("Open your app here:", ngrok.connect(8501).public_url)

---

## What to take away

**Use a single Messages API call when you can specify the task.** Extraction,
classification, scoring against a fixed rubric — one call is cheaper, faster and
easier to test.

**Reach for the Agent SDK when the steps aren't knowable in advance.** The agent
reads files, decides what to look at next, and produces an artefact. You pay for
that in latency and tokens, so it has to be earning its place.

**Never hardcode an API key.** Both parts of this lab took the key at run time —
`getpass` in the notebook, a password field in the app. Nothing was written to
disk, and nothing went into the repository.